# Knowledge graphs: a hands-on implementation

**Track:** Search · **Guide:** [`048-knowledge-graphs.md`](../guides/concepts/048-knowledge-graphs.md)

A knowledge graph represents entities as nodes and typed relationships as edges.

> **Intuition:** A map labels places and the roads connecting them; the connection itself has meaning.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Knowledge graphs]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Trace which product versions, components, and advisories are connected during an incident review.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Model facts as typed triples

A knowledge graph stores subject–relation–object facts with explicit relation names.


In [ ]:
triples = [
    ("Battery-B7", "installed_in", "Device-D1"),
    ("Device-D1", "uses_firmware", "Firmware-2.4"),
    ("Firmware-2.4", "affected_by", "Advisory-A9"),
    ("Battery-B7", "manufactured_by", "Maker-M3"),
]

## 2. Query the graph

A simple traversal illustrates graph queries; graph databases add indexes, query languages, and constraints.


In [ ]:
def neighbors(entity, relation=None):
    return [
        (edge, target)
        for source, edge, target in triples
        if source == entity and (relation is None or edge == relation)
    ]

print("Device components:", neighbors("Battery-B7"))
device = neighbors("Battery-B7", "installed_in")[0][1]
firmware = neighbors(device, "uses_firmware")[0][1]
print("firmware advisories:", neighbors(firmware, "affected_by"))

## 3. Answer a multi-hop question

Relationships enable questions that require combining several facts; validate entity resolution and edge provenance in real graphs.


In [ ]:
def advisories_for_component(component):
    devices = [target for relation, target in neighbors(component, "installed_in")]
    firmware = [
        fw for device_id in devices
        for relation, fw in neighbors(device_id, "uses_firmware")
    ]
    return [
        advisory
        for fw in firmware
        for relation, advisory in neighbors(fw, "affected_by")
    ]

print(advisories_for_component("Battery-B7"))

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/048-knowledge-graphs.md) and the
[learning roadmap](../guides/ROADMAP.md).
